# Project Sentinel – Gold Layer

## Objective

This notebook creates business-ready analytical datasets from the Silver layer. It computes key performance indicators (KPIs) and stores aggregated results in Delta tables for reporting and analytics.

### Features
- Revenue Analysis
- Top Products
- Top Customers
- Payment Distribution
- Delta Lake
- Business KPI Generation

### Execution Flow

Input:
- Silver Delta Table

Processing:
- Aggregate business metrics
- Compute KPIs
- Generate analytical datasets

Output:
- Gold Delta Tables

Next Notebook:
05_SQL_Analytics

In [0]:
BASE_PATH = "/Volumes/workspace/default/project_sentinel"

SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"

print("Silver:", SILVER_PATH)
print("Gold:", GOLD_PATH)

Silver: /Volumes/workspace/default/project_sentinel/silver
Gold: /Volumes/workspace/default/project_sentinel/gold


In [0]:
from pyspark.sql.functions import *

In [0]:

silver_df = spark.read.format("delta").load(SILVER_PATH)

print("Silver Records:", silver_df.count())

display(silver_df.limit(10))

Silver Records: 4489


transaction_id,transactional_date,product_id,customer_id,payment,credit_card,loyalty_card,cost,quantity,price,_rescued_data,ingestion_time
5,2021-05-04 05:45:00,P0058,5,mastercard,5108752372298261,T,11.38,2,12.39,null,2026-07-08T17:04:28.822Z
27,2021-05-05 20:09:00,P0622,5,visa,4041596972700199,F,13.87,1,14.89,null,2026-07-08T17:04:28.822Z
45,2021-05-08 01:48:00,P0431,4,americanexpress,374283964832018,F,10.67,4,11.59,null,2026-07-08T17:04:28.822Z
46,2021-05-08 03:04:00,P0170,10,americanexpress,374288240835982,F,8.13,4,8.89,null,2026-07-08T17:04:28.822Z
49,2021-05-08 13:43:00,P0661,3,visa,4041593023791707,F,8.21,4,8.99,null,2026-07-08T17:04:28.822Z
56,2021-05-08 22:40:00,P0035,6,visa,4041598082914470,F,11.76,3,13.09,null,2026-07-08T17:04:28.822Z
57,2021-05-08 23:32:00,P0268,7,americanexpress,374288707197975,T,11.93,1,13.39,null,2026-07-08T17:04:28.822Z
102,2021-05-12 18:32:00,P0545,8,mastercard,5108757985493662,T,6.62,1,7.89,null,2026-07-08T17:04:28.822Z
109,2021-05-13 07:17:00,P0423,8,mastercard,5108754133243636,F,14.04,3,14.89,null,2026-07-08T17:04:28.822Z
122,2021-05-15 01:03:00,P0305,6,americanexpress,374283437595721,F,8.74,3,9.99,null,2026-07-08T17:04:28.822Z


In [0]:
silver_df.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- transactional_date: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- payment: string (nullable = true)
 |-- credit_card: string (nullable = true)
 |-- loyalty_card: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)



In [0]:
total_revenue = (
    silver_df
    .select(
        sum(col("price") * col("quantity"))
        .alias("Total_Revenue")
    )
)

display(total_revenue)

Total_Revenue
105122.24999999885


In [0]:
total_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/total_revenue")

In [0]:
top_products = (
    silver_df
    .groupBy("product_id")
    .agg(
        sum("quantity").alias("Total_Quantity")
    )
    .orderBy(desc("Total_Quantity"))
    .limit(10)
)

display(top_products)

product_id,Total_Quantity
P0486,45
P0467,45
P0339,44
P0395,40
P0305,38
P0387,36
P0437,36
P0440,36
P0463,36
P0410,35


In [0]:
top_products.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/top_products")

In [0]:
top_customers = (
    silver_df
    .groupBy("customer_id")
    .agg(
        sum(col("price") * col("quantity"))
        .alias("Revenue")
    )
    .orderBy(desc("Revenue"))
    .limit(10)
)

display(top_customers)

customer_id,Revenue
6,21515.899999999987
5,18977.790000000037
7,18900.860000000044
4,12594.320000000002
8,11683.800000000012
3,7144.899999999994
9,6503.279999999995
2,2814.399999999999
10,2769.5399999999995
1,918.2300000000002


In [0]:
dbutils.fs.rm(GOLD_PATH, True)

True

In [0]:
top_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/top_customers")

In [0]:
payment_distribution = (
    silver_df
    .groupBy("payment")
    .count()
)

display(payment_distribution)

payment,count
americanexpress,1421
mastercard,1426
visa,1405
null,177


In [0]:
payment_distribution.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}/payment_distribution")

In [0]:
display(dbutils.fs.ls(GOLD_PATH))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/project_sentinel/gold/payment_distribution/,payment_distribution/,0,1783573149804
dbfs:/Volumes/workspace/default/project_sentinel/gold/top_customers/,top_customers/,0,1783573149804
dbfs:/Volumes/workspace/default/project_sentinel/gold/top_products/,top_products/,0,1783573149804
dbfs:/Volumes/workspace/default/project_sentinel/gold/total_revenue/,total_revenue/,0,1783573149804


---
### Notebook Completed Successfully ✅

**Project:** Project Sentinel

**Architecture:** Medallion Architecture

**Technology Stack:** Databricks • PySpark • Delta Lake • Auto Loader • Structured Streaming